# Isolate Mapping Statistics

All fastp-passed paired and orphan reads are counted. The priority stages are mutually exclusive read assignments; they do not prove exclusive biological origins. Full-read host-identification and all-assembled mappings, when requested, are independent comparisons.

In [ ]:
# import necessary modules
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

#------------------------------------------
#Create a list of colors
colors_rarefaction=sns.color_palette("colorblind", n_colors=len(list(snakemake.params.samples)))
#Create a LinearSegmentedColormap object
cmap1=LinearSegmentedColormap.from_list("my_colormap", sns.color_palette("colorblind", n_colors=5))


sns.set_style("ticks",{'axes.grid' : True})
sns.set_palette("colorblind")

plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["axes.titlepad"] = 20

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams["axes.titlesize"] = 30
plt.rcParams['axes.labelsize'] = 23.5
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18
plt.rcParams['legend.fontsize'] = 18
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Liberation Sans']
plt.rcParams['text.usetex'] = False

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams["savefig.dpi"]=300

In [ ]:
from pathlib import Path
from IPython.display import display, Image
SAMPLES=list(snakemake.params.samples)
stages=list(snakemake.params.stages)
tables=[pd.read_csv(path, sep="\t") for path in snakemake.input.accounting]
accounting=pd.concat(tables, ignore_index=True)
final=accounting[accounting["stage"].eq(stages[-1])].copy()
final["stage"]="07_unexplained"
final["assigned_reads"]=final["remaining_reads"]
final["percent_original"]=final["remaining_percent_original"]
final["percent_entering"]=np.where(final["remaining_reads"] > 0, 100, 0)
accounting=pd.concat([accounting, final], ignore_index=True)
accounting.to_csv(snakemake.output.isolate_table[0], sep="\t", index=False)
display(accounting)
matrix=accounting.pivot(index="sample", columns="stage", values="percent_original").reindex(index=SAMPLES, columns=stages+["07_unexplained"]).fillna(0)


In [ ]:
fig, ax=plt.subplots(figsize=(max(10, 0.3 * len(SAMPLES)), 6))
matrix.plot.bar(stacked=True, ax=ax, color=sns.color_palette("colorblind", 7), width=0.85)
ax.set_xlabel("Sample")
ax.set_ylabel("All QC-passed reads (%)")
ax.set_ylim(0, 100)
ax.legend(title="Priority assignment", bbox_to_anchor=(1.02, 1), loc="upper left")
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(snakemake.output.isolate_png[0], dpi=300, bbox_inches="tight")
fig.savefig(snakemake.output.isolate_svg[0], bbox_inches="tight")
plt.close(fig)
display(Image(filename=snakemake.output.isolate_png[0]))


## Optional Independent All-Assembled Mapping

This is not part of the residual-read partition. Its percentage must not be added to the seven priority-stage fractions.

In [ ]:
independent=[]
for sample, path in zip(SAMPLES, snakemake.input.all_assembled_flagstats):
    primary_mapped=0
    with open(path) as handle:
        for line in handle:
            if " primary mapped (" in line:
                primary_mapped=int(line.split()[0])
    total=accounting[accounting["sample"].eq(sample)]["original_reads"].iloc[0]
    independent.append(dict(sample=sample, mapped_reads=primary_mapped,
                            percent_full_qc_reads=100 * primary_mapped / total if total else 0))
independent=pd.DataFrame(independent, columns=["sample", "mapped_reads", "percent_full_qc_reads"])
display(independent)
parts=["<html><head><meta charset='utf-8'></head><body><h1>Isolate Mapping Statistics</h1>",
       "<p>Full QC-passed reads, counted once per priority stage. Single-mate, discordant and non-unique evidence is retained.</p>",
       accounting.to_html(index=False),
       f"<img src='{Path(snakemake.output.isolate_png[0]).name}' style='max-width:100%'>",
       "<h2>Optional independent all-assembled comparison</h2>", independent.to_html(index=False), "</body></html>"]
Path(snakemake.output.mapping_stats_html).write_text("\n".join(parts))
